<h2 align="center" style="color:blue">Codebasics Python Course: Python for AI</h2>

**Background:** Our product team is building a smart travel assistant for our booking platform. Users often share their preferences across multiple messages.

**Problem Statement:** Your task is to build an interactive travel planner that maintains conversational memory, calls an LLM API dynamically, handles errors gracefully, and finally exports the itinerary into a strict Pydantic structured output.

### Task 1: Define the Pydantic Schema

Create a Pydantic `BaseModel` called `TravelItinerary` that includes:
- `destination` (str)
- `duration_days` (int)
- `budget_category` (str)
- `activities` (List of strings)
- `daily_itinerary` (List of objects containing `day` and `plan`)

*Hint: You may need a secondary model for the daily plan.*

In [2]:
# Write your code here to define the Pydantic models
from pydantic import BaseModel

class DailyPlan(BaseModel):
    day: int
    plan: str

class TravelItinerary(BaseModel):
    destination: str
    duration_days: int
    budget_category: str
    activities: list[str]
    daily_itinerary: list[DailyPlan]

### Task 2: Configure the API and Initialize Memory

Set up your LLM client (e.g., Google Gemini) and initialize an empty list called `chat_history` to store the conversation state.

In [ ]:
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()  # Load environment variables from .env file

client = genai.Client()  # Initialize the Google GenAI client

# Write your code here to set up the API and memory list
chat_history = []



### Task 3: Build the Chat Loop with Structured Output Trigger

Create a `while True` loop that:
1. Takes user input.
2. If the user types `DONE`, wraps the API call in a `try...except` block, enforces the `TravelItinerary` schema on the output, prints the structured JSON, and breaks the loop.
3. Otherwise, appends the user input to `chat_history`, calls the LLM, prints the response, and appends the model's response to `chat_history`.

In [11]:
# Write your code here to build the while loop, handle exceptions, and enforce structured output
def chat(user_message):
    chat_history.append({"role": "user", "parts": [{"text": user_message}]})

    try:
        response = client.models.generate_content(
            model="gemini-3.5-flash",
            contents=chat_history,
            config = types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=TravelItinerary
                )
        )
        reply = response.text
    except Exception as e:
        print(f"Error during chat generation: {e}")
        return "Sorry, I couldn't process your request at the moment."
    
    chat_history.append({"role": "assistant", "parts": [{"text": reply}]})
    return reply

In [12]:
chat("I want to plan a 5-day trip to Paris with a budget of $2000 and include activities like Eiffel Tower visit, Louvre Museum tour, and Seine River cruise.")

'{"destination":"Paris","duration_days":5,"budget_category":"$2000","activities":["Eiffel Tower visit","Louvre Museum tour","Seine River cruise"],"daily_itinerary":[{"day":1,"plan":"Arrive in Paris, check into your accommodation, and take a relaxing stroll along the Champs-Elysees."},{"day":2,"plan":"Visit the Eiffel Tower in the morning, explore the Trocadero area, and embark on a Seine River cruise at sunset."},{"day":3,"plan":"Take a guided Louvre Museum tour in the morning and spend the afternoon relaxing in the Tuileries Garden."},{"day":4,"plan":"Explore the artistic district of Montmartre, visit Sacre-Coeur Basilica, and wander through the Latin Quarter."},{"day":5,"plan":"Enjoy breakfast at a traditional Parisian cafe, do some souvenir shopping, and prepare for departure."}]}'

In [13]:
for message in chat_history:
    print(f"{message['role'].capitalize()}: {message['parts'][0]['text']}")

User: I want to plan a 5-day trip to Paris with a budget of $2000 and include activities like Eiffel Tower visit, Louvre Museum tour, and Seine River cruise.
User: I want to plan a 5-day trip to Paris with a budget of $2000 and include activities like Eiffel Tower visit, Louvre Museum tour, and Seine River cruise.
Assistant: {"destination":"Paris","duration_days":5,"budget_category":"Moderate","activities":["Eiffel Tower visit","Louvre Museum tour","Seine River cruise"],"daily_itinerary":[{"day":1,"plan":"Arrival in Paris, check into hotel, and take a relaxed stroll along the Champs-Elysees."},{"day":2,"plan":"Visit the Eiffel Tower in the morning, explore the Trocadero area, and enjoy an evening Seine River cruise."},{"day":3,"plan":"Spend the day at the Louvre Museum and take a walk through the nearby Tuileries Garden."},{"day":4,"plan":"Explore Montmartre, visit the Sacre-Coeur Basilica, and wander through the Latin Quarter."},{"day":5,"plan":"Enjoy a final French breakfast, do some